In [1]:
from glob import glob
from moviepy import VideoFileClip, concatenate_videoclips
import os
import numpy as np
import pandas as pd

In [2]:
# set variables
data_dir = '/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out'
patient = 'YFS'
save_path = f'{data_dir}/{patient}/all_convo_recording'

In [257]:
# get all clips for patient that have 'video_out.avi'
recording_dirs = glob(f'{data_dir}/{patient}/recording*/video/recording*/neural/23512014/synced_video/neural*/pyavi/video_out.avi')

In [259]:
recording_dirs = ["/".join(path.split("/")[:-2]) for path in recording_dirs]

In [260]:
# get paths to videos
video_paths = [f"{path}/pyavi/video.avi" for path in recording_dirs]

In [261]:
# get paths to speaker scoring
speaker_paths = [f"{path}/pywork/scores.pckl" for path in recording_dirs]

In [271]:
# load speaker scores
import dill as pickle
speaker_scores = []
for path in speaker_paths:
    with open(path, 'rb') as f:
        score = pickle.load(f)
        speaker_scores.append(score)
        

In [280]:
# get paths to tracks
track_paths = [f"{path}/pywork/tracks.pckl" for path in recording_dirs]

In [281]:
# load speaker tracks
import dill as pickle
tracks = []
for path in track_paths:
    with open(path, 'rb') as f:
        track = pickle.load(f)
        tracks.append(track)
        

In [282]:
face_paths = [f"{path}/pywork/faces.pckl" for path in recording_dirs]

In [283]:
# load faces
import dill as pickle
faces = []
for path in face_paths:
    with open(path, 'rb') as f:
        face = pickle.load(f)
        faces.append(face)
        

In [284]:
FPS=25.0
def moving_average_nan(x: np.ndarray, window: int) -> np.ndarray:
    """
    Moving average that ignores NaNs.
    """
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x).astype(float)
    x_filled = np.nan_to_num(x, nan=0.0)

    kernel = np.ones(window, dtype=float)
    num = np.convolve(x_filled, kernel, mode="same")
    den = np.convolve(valid, kernel, mode="same")

    out = np.full_like(x, np.nan, dtype=float)
    keep = den > 0
    out[keep] = num[keep] / den[keep]
    return out


def merge_ldasd_tracks(
    tracks,
    scores,
    fps: float = 25.0,
    smooth_ms: float = 400.0,
    threshold: float = 0.0,
):
    """
    Returns a dataframe with one row per global frame:
      - frame_idx
      - time_sec
      - raw_score
      - smooth_score
      - speaking
    """
    if len(tracks) != len(scores):
        raise ValueError(f"tracks ({len(tracks)}) and scores ({len(scores)}) length mismatch")

    # Determine total frame span from all tracks
    max_frame = -1
    for tr in tracks:
        frames = np.asarray(tr["track"]["frame"])
        if len(frames) == 0:
            continue
        max_frame = max(max_frame, int(frames.max()))

    if max_frame < 0:
        raise ValueError("No frames found in tracks")

    n_frames = max_frame + 1

    # One row per track, one column per global frame
    score_mat = np.full((len(tracks), n_frames), np.nan, dtype=float)

    for tidx, (tr, sc) in enumerate(zip(tracks, scores)):
        frames = np.asarray(tr["track"]["frame"], dtype=int)
        sc = np.asarray(sc, dtype=float)

        # Guard against slight length mismatch
        n = min(len(frames), len(sc))
        frames = frames[:n]
        sc = sc[:n]

        score_mat[tidx, frames] = sc

    # Since you assume only one face in view, max is a clean merge rule
    global_raw = np.nanmax(score_mat, axis=0)

    # If a frame had no track at all, nanmax returns warning/NaN behavior
    # Make sure fully empty columns stay NaN
    has_any = np.any(np.isfinite(score_mat), axis=0)
    global_raw[~has_any] = np.nan

    # Smooth over chosen window
    window = max(1, int(round((smooth_ms / 1000.0) * fps)))
    if window % 2 == 0:
        window += 1  # odd window is a bit nicer for centering

    global_smooth = moving_average_nan(global_raw, window=window)

    # Threshold only where we have data
    speaking = np.zeros(n_frames, dtype=int)
    speaking[has_any] = (global_smooth[has_any] >= threshold).astype(int)

    df = pd.DataFrame({
        "frame_idx": np.arange(n_frames, dtype=int),
        "time_sec": np.arange(n_frames, dtype=float) / fps,
        "time_ms": (np.arange(n_frames, dtype=float) / fps) * 1000.0,
        "raw_score": global_raw,
        "smooth_score": global_smooth,
        "has_face": has_any.astype(int),
        "speaking": speaking,
    })

    return df


def speaking_segments_from_df(df: pd.DataFrame, min_duration_ms: float = 200.0) -> pd.DataFrame:
    """
    Convert framewise speaking labels into contiguous speaking segments.
    """
    speaking = df["speaking"].to_numpy().astype(int)
    time_sec = df["time_sec"].to_numpy()

    segments = []
    in_seg = False
    start_idx = None

    for i, val in enumerate(speaking):
        if val == 1 and not in_seg:
            in_seg = True
            start_idx = i
        elif val == 0 and in_seg:
            end_idx = i - 1
            segments.append((start_idx, end_idx))
            in_seg = False

    if in_seg:
        segments.append((start_idx, len(speaking) - 1))

    rows = []
    min_duration_sec = min_duration_ms / 1000.0

    for s, e in segments:
        start_sec = time_sec[s]
        end_sec = time_sec[e] + (1.0 / FPS)  # make end exclusive-ish
        dur = end_sec - start_sec
        if dur >= min_duration_sec:
            rows.append({
                "start_frame": s,
                "end_frame": e,
                "start_sec": start_sec,
                "end_sec": end_sec,
                "start_ms": start_sec * 1000.0,
                "end_ms": end_sec * 1000.0,
                "duration_sec": dur,
            })

    return pd.DataFrame(rows)

In [285]:
# get utterance df and speaking segments df for all videos
utterance_dfs = []

for i in range(len(tracks)):
    try:
        df = merge_ldasd_tracks(
            tracks=tracks[i],
            scores=speaker_scores[i],
            fps=25.0,
            smooth_ms=400.0,   # try 200, 300, 400, 500
            threshold=0.0,     # repo uses >= 0 for speaking
        ) 
        utterance_dfs.append(df)
    except Exception as e:
        print(e)
        utterance_dfs.append(None)


No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks


/tmp/ipykernel_435372/4083256028.py:66: RuntimeWarning: All-NaN slice encountered
  global_raw = np.nanmax(score_mat, axis=0)


No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks
No frames found in tracks


In [286]:
speaker_segs = []

for i in range(len(utterance_dfs)):
    try:
        segs = speaking_segments_from_df(utterance_dfs[i], min_duration_ms=200.0)
        speaker_segs.append(segs)
    except Exception as e:
        print(e)
        speaker_segs.append(pd.DataFrame())

'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not subscriptable
'NoneType' object is not 

In [290]:
# get clips to stitch together
stitch_clips = []
for idx, seg in enumerate(speaker_segs):
    if seg.empty:
        continue
    print("starting seg", idx)
    video_path = video_paths[idx]
    video = VideoFileClip(video_path)
    for jdx, row in seg.iterrows():
        clip = video.subclipped(row.start_sec, row.end_sec)
        stitch_clips.append(clip)



starting seg 3
starting seg 4
starting seg 5
starting seg 6
starting seg 12
starting seg 13
starting seg 14
starting seg 15
starting seg 26
starting seg 27
starting seg 30
starting seg 31
starting seg 32
starting seg 34
starting seg 35
starting seg 39
starting seg 41
starting seg 42
starting seg 49
starting seg 54
starting seg 57
starting seg 58
starting seg 61
starting seg 62
starting seg 63
starting seg 65
starting seg 68
starting seg 69
starting seg 83
starting seg 88
starting seg 92
starting seg 100
starting seg 102
starting seg 103
starting seg 106
starting seg 107
starting seg 108
starting seg 109
starting seg 111
starting seg 115
starting seg 116
starting seg 118
starting seg 119
starting seg 120
starting seg 121
starting seg 122
starting seg 132
starting seg 134
starting seg 137
starting seg 138
starting seg 140
starting seg 141
starting seg 142
starting seg 143
starting seg 145
starting seg 150
starting seg 151
starting seg 152
starting seg 154
starting seg 155
starting seg 15

In [ ]:
# concatenate into signle video and save
final_clip = concatenate_videoclips(stitch_clips)
final_clip.write_videofile(f"{save_path}/all_speaker_clips.mp4", audio_codec="aac")


MoviePy - Building video /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out/YFS/all_convo_recording/all_speaker_clips.mp4.
MoviePy - Writing audio in all_speaker_clipsTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out/YFS/all_convo_recording/all_speaker_clips.mp4



frame_index:   1%|          | 1286/230610 [00:43<2:22:44, 26.78it/s, now=None]